<a href="https://colab.research.google.com/github/TahaNawazkhan/Artificial-intelligence-/blob/main/lab%201%20task%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

ACTIONS = ["Left", "Right", "Suck", "NoOp"]


class ReflexAgent:
    """Simple reflex agent: no memory, never stops."""

    def reset(self):
        pass

    def act(self, percept):
        location, status = percept
        if status == "Dirty":
            return "Suck"
        return "Right" if location == "A" else "Left"


class MemoryReflexAgent:
    """Reflex agent that remembers which locations are clean and stops (NoOp) when both are."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.known_clean = set()

    def act(self, percept):
        location, status = percept
        if status == "Dirty":
            self.known_clean.add(location)      # Suck always succeeds, so remember it
            return "Suck"
        self.known_clean.add(location)          # seen clean -> remember it
        if len(self.known_clean) == 2:          # both locations clean -> stop
            return "NoOp"
        return "Right" if location == "A" else "Left"


def random_initial_state():
    return {
        "loc": random.choice(["A", "B"]),
        "dirt": {"A": random.choice(["Clean", "Dirty"]),
                 "B": random.choice(["Clean", "Dirty"])},
    }


def apply_action(state, action):
    if action == "Suck":
        state["dirt"][state["loc"]] = "Clean"
    elif action == "Left":
        state["loc"] = "A"
    elif action == "Right":
        state["loc"] = "B"
    # NoOp: nothing changes


def run(agent, initial_state, steps=20, move_penalty=1.0):
    """+1 per clean square each step, minus move_penalty for every Left/Right action."""
    agent.reset()
    state = {"loc": initial_state["loc"], "dirt": dict(initial_state["dirt"])}
    score, moves = 0.0, 0
    for _ in range(steps):
        percept = (state["loc"], state["dirt"][state["loc"]])
        action = agent.act(percept)
        apply_action(state, action)
        if action in ("Left", "Right"):
            moves += 1
            score -= move_penalty
        score += sum(1 for s in state["dirt"].values() if s == "Clean")
    return score, moves


if __name__ == "__main__":
    random.seed(42)
    STEPS, TRIALS, PENALTY = 20, 10, 1.0
    reflex, memory = ReflexAgent(), MemoryReflexAgent()
    initial_states = [random_initial_state() for _ in range(TRIALS)]

    print(f"20 steps, move penalty = {PENALTY}\n")
    print(f"{'Trial':<6}{'Start (loc, A, B)':<22}{'Reflex':>8}{'(moves)':>9}{'Memory':>9}{'(moves)':>9}")
    r_scores, m_scores = [], []
    for i, init in enumerate(initial_states, 1):
        r, rm = run(reflex, init, STEPS, PENALTY)
        m, mm = run(memory, init, STEPS, PENALTY)
        r_scores.append(r)
        m_scores.append(m)
        start = f"{init['loc']}, {init['dirt']['A']}, {init['dirt']['B']}"
        print(f"{i:<6}{start:<22}{r:>8.0f}{rm:>9}{m:>9.0f}{mm:>9}")

    print(f"\nAverage simple reflex agent: {sum(r_scores) / TRIALS:.1f}")
    print(f"Average memory reflex agent: {sum(m_scores) / TRIALS:.1f}")

    # Effect of the penalty size (averaged over 1000 random initial states)
    print("\nAverage score over 1000 random initial states:")
    print(f"{'Move penalty':<14}{'Reflex':>9}{'Memory':>9}")
    many = [random_initial_state() for _ in range(1000)]
    for p in (0.0, 0.5, 1.0, 2.0):
        r_avg = sum(run(reflex, s, STEPS, p)[0] for s in many) / len(many)
        m_avg = sum(run(memory, s, STEPS, p)[0] for s in many) / len(many)
        print(f"{p:<14}{r_avg:>9.2f}{m_avg:>9.2f}")

20 steps, move penalty = 1.0

Trial Start (loc, A, B)       Reflex  (moves)   Memory  (moves)
1     A, Clean, Dirty             20       19       38        1
2     A, Clean, Clean             20       20       39        1
3     A, Clean, Dirty             20       19       38        1
4     A, Clean, Clean             20       20       39        1
5     A, Clean, Clean             20       20       39        1
6     A, Dirty, Clean             21       19       39        1
7     B, Dirty, Clean             20       19       38        1
8     A, Dirty, Dirty             20       18       37        1
9     B, Clean, Clean             20       20       39        1
10    B, Clean, Clean             20       20       39        1

Average simple reflex agent: 20.1
Average memory reflex agent: 38.5

Average score over 1000 random initial states:
Move penalty     Reflex   Memory
0.0               39.23    39.23
0.5               29.74    38.73
1.0               20.24    38.23
2.0              